# 08 · Food Relief Priority Index

FRPI = 0.6 · calibrated P(High ∪ Severe) + 0.4 · food-fragility score. Fragility scalers are fit on TRAIN only. Ranks and the [0,1] display score are computed within each storm, and maps are rendered for every TEST hurricane.

In [1]:
import sys, os
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', 60)
from config import (
    DATA_PATHS, HURRICANE_META, STATES_IN_SCOPE,
    TARGET_COL, TARGET_CLASS_COL, FEATURE_GROUPS,
    RANDOM_STATE, SEVERITY_BINS, SEVERITY_LABELS,
)
RAW = DATA_PATHS['raw']; INTERIM = DATA_PATHS['interim']
PROC = DATA_PATHS['processed']; MODELS = DATA_PATHS['models']
OUT = DATA_PATHS['outputs']


In [2]:
import joblib, folium
import geopandas as gpd
from src.priority_index import fit_fragility_scalers, apply_fragility, priority_index, save_scalers
df = pd.read_csv(PROC / 'abt.csv', dtype={'zip_code': str}).dropna(subset=[TARGET_CLASS_COL])
pkg = joblib.load(MODELS / 'best_classifier.pkl')
calibrator, le, features = pkg['calibrator'], pkg['label_encoder'], pkg['features']

In [3]:
# Fit fragility scalers on TRAIN only
scalers = fit_fragility_scalers(df[df['train_test_split'] == 'TRAIN'])
save_scalers(scalers, MODELS / 'fragility_scalers.pkl')
df = apply_fragility(df, scalers)

In [4]:
# Calibrated probabilities for TEST; same row order as notebook 07 / shap_values.npy
te = df[df['train_test_split'] == 'TEST'].reset_index(drop=True)
proba = calibrator.predict_proba(te[features])
parts = []
for hn, idx in te.groupby('hurricane_name').groups.items():
    part = priority_index(te.loc[idx], proba[np.asarray(idx)], list(le.classes_))
    part.index = idx
    parts.append(part)
te_prio = pd.concat(parts).sort_index()
te_prio.to_csv(OUT / 'priority_rankings.csv', index=False)
print('saved priority_rankings.csv', te_prio.shape)

saved priority_rankings.csv (2587, 72)


/Users/anees/.pyenv/versions/3.11.8/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


## Validation — what each storm's top decile contains

In [5]:
rows = []
for hn, g in te_prio.groupby('hurricane_name'):
    k = max(1, round(0.10 * len(g)))
    top = g.nsmallest(k, 'priority_rank')
    rows.append({'hurricane': hn, 'zips': len(g), 'top_decile': k,
                 'true_high_or_severe': top[TARGET_CLASS_COL].isin(['High', 'Severe']).mean(),
                 'storm_base_rate': g[TARGET_CLASS_COL].isin(['High', 'Severe']).mean(),
                 'true_low': (top[TARGET_CLASS_COL] == 'Low').mean(),
                 'food_desert': top['food_desert_flag'].mean()})
summary = pd.DataFrame(rows).set_index('hurricane')
summary.loc['mean'] = summary.mean()
summary.round(3)

,zips,top_decile,true_high_or_severe,storm_base_rate,true_low,food_desert
hurricane,,,,,,
Helene,329.0,33.0,0.939,0.672,0.030,0.364
Ian,739.0,74.0,0.865,0.516,0.054,0.365
Ida,343.0,34.0,1.000,0.647,0.000,0.412
Idalia,269.0,27.0,0.963,0.257,0.037,0.259
Milton,907.0,91.0,0.813,0.398,0.000,0.308
mean,517.4,51.8,0.916,0.498,0.024,0.341


## Folium choropleth maps — every TEST hurricane

In [6]:
zcta = gpd.read_file(next((RAW / 'zcta').rglob('*.shp'))).to_crs('EPSG:4326')
zcta['zip_code'] = zcta['ZCTA5CE20'].astype(str).str.zfill(5)
TOP_N = 200  # render only the highest-priority ZIPs per hurricane
for hn in sorted(te_prio['hurricane_name'].unique()):
    sub = te_prio[te_prio['hurricane_name'] == hn].nsmallest(TOP_N, 'priority_rank')
    z = zcta.merge(sub[['zip_code', 'priority_rank', 'priority_index_norm',
                        'prob_high_or_severe', 'food_fragility_score']], on='zip_code')
    if z.empty:
        print('no zips for', hn); continue
    z['geometry'] = z.geometry.simplify(tolerance=0.001, preserve_topology=True)
    cent = z.geometry.union_all().centroid
    m = folium.Map(location=[cent.y, cent.x], zoom_start=7, tiles='cartodbpositron')
    folium.Choropleth(
        geo_data=z, data=z, columns=['zip_code', 'priority_index_norm'],
        key_on='feature.properties.zip_code', fill_color='YlOrRd', fill_opacity=0.7,
        line_opacity=0.2, legend_name=f'Priority index — {hn} (top {TOP_N})',
    ).add_to(m)
    folium.GeoJson(
        z, name='details', style_function=lambda _: {'fillOpacity': 0, 'color': 'transparent'},
        tooltip=folium.GeoJsonTooltip(
            fields=['zip_code', 'priority_rank', 'priority_index_norm',
                    'prob_high_or_severe', 'food_fragility_score'],
            aliases=['ZIP', 'Rank', 'Priority (norm)', 'P(High∪Severe)', 'Fragility'],
            localize=True),
    ).add_to(m)
    out = OUT / f'priority_map_{hn.lower()}.html'
    m.save(str(out)); print(f'saved {out}  ({len(z)} ZIPs)')

/Users/anees/.pyenv/versions/3.11.8/lib/python3.11/site-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


saved /Users/anees/Documents/Github/hurricane-food-relief-ml/outputs/priority_map_helene.html  (200 ZIPs)
saved /Users/anees/Documents/Github/hurricane-food-relief-ml/outputs/priority_map_ian.html  (200 ZIPs)


/Users/anees/.pyenv/versions/3.11.8/lib/python3.11/site-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore
/Users/anees/.pyenv/versions/3.11.8/lib/python3.11/site-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


saved /Users/anees/Documents/Github/hurricane-food-relief-ml/outputs/priority_map_ida.html  (200 ZIPs)


/Users/anees/.pyenv/versions/3.11.8/lib/python3.11/site-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


saved /Users/anees/Documents/Github/hurricane-food-relief-ml/outputs/priority_map_idalia.html  (200 ZIPs)
saved /Users/anees/Documents/Github/hurricane-food-relief-ml/outputs/priority_map_milton.html  (200 ZIPs)


/Users/anees/.pyenv/versions/3.11.8/lib/python3.11/site-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore
